# ImageNet Attention Heatmap (patch size = 4)

This notebook loads an ImageNet image via Hugging Face `datasets`, preprocesses it (resize and crop to multiples of patch size 4), converts to patch embeddings, computes self-attention using `torch.nn.MultiheadAttention`, and overlays the attention map of a chosen query patch onto the image.


In [1]:
# Imports and configuration
import warnings
warnings.filterwarnings("ignore")

import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
import matplotlib.pyplot as plt

# Config
patch_size = 4
max_side = 128
seed = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_ = torch.manual_seed(seed)

print({
    "device": str(device),
    "patch_size": patch_size,
    "max_side": max_side,
})


KeyboardInterrupt: 

In [ ]:
# Load a sample from ImageNet-1k (fallback: Imagenette)
from typing import Tuple

try:
    ds = load_dataset("imagenet-1k", split="train", trust_remote_code=True)
    dataset_name = "imagenet-1k"
except Exception as e:
    print("Falling back to Imagenette due to:", e)
    try:
        ds = load_dataset("imagenette", split="train")
        dataset_name = "imagenette"
    except Exception as e2:
        print("Failed to load Imagenette as well:", e2)
        raise

print(f"Dataset: {dataset_name}, length: {len(ds)}")

sample_index = 0  # change this to visualize another sample
example = ds[int(sample_index)]

# Most HF vision datasets return PIL Images already under key 'image'
img_pil: Image.Image = example["image"].convert("RGB")
print("Original size:", img_pil.size)

plt.figure(figsize=(3, 3))
plt.imshow(img_pil)
plt.axis("off")
plt.title(f"{dataset_name} sample #{sample_index}")
plt.show()


In [ ]:
# Preprocess: ResizeMaxSide and crop to multiple of patch size

class ResizeMaxSide:
    def __init__(self, max_side: int, interpolation=Image.BILINEAR):
        self.max_side = max_side
        self.interpolation = interpolation
    def __call__(self, img: Image.Image) -> Image.Image:
        width, height = img.size
        max_dim = max(width, height)
        if max_dim > self.max_side:
            scale = self.max_side / max_dim
            new_width = int(width * scale)
            new_height = int(height * scale)
            img = img.resize((new_width, new_height), self.interpolation)
        return img

resize = ResizeMaxSide(max_side)

# Resize (keeping aspect), then convert to tensor
img_resized_pil = resize(img_pil)
img_tensor = transforms.ToTensor()(img_resized_pil)  # [C, H, W] in [0,1]

# Ensure 3 channels
if img_tensor.shape[0] > 3:
    img_tensor = img_tensor[:3]
elif img_tensor.shape[0] == 1:
    img_tensor = img_tensor.repeat(3, 1, 1)

C, H, W = img_tensor.shape

# Crop H, W to be multiples of patch_size
H_crop = H - (H // patch_size) * patch_size
W_crop = W - (W // patch_size) * patch_size
if H_crop > 0 or W_crop > 0:
    img_tensor = img_tensor[:, : H - H_crop, : W - W_crop]

C, H, W = img_tensor.shape
nh, nw = H // patch_size, W // patch_size
print({"resized": img_resized_pil.size, "cropped_HW": (H, W), "nh_nw": (nh, nw)})

# Keep a numpy image for plotting overlays
img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


In [ ]:
# Patchify with p=4 and build 2D sinusoidal positional embeddings

def patchify(x: torch.Tensor, p: int) -> torch.Tensor:
    # C, H, W -> T, D (T = nh*nw, D = p*p*C)
    c, h, w = x.shape
    nh, nw = h // p, w // p
    x = x.view(c, nh, p, nw, p)
    x = x.permute(1, 3, 2, 4, 0).contiguous()
    x = x.view(nh * nw, p * p * c)
    return x

# Positional embeddings (2D sin-cos)
def get_1d_sincos_pos_embed(length: int, dim: int) -> torch.Tensor:
    assert dim % 2 == 0
    pos = torch.arange(length).float().unsqueeze(1)  # [L,1]
    div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))  # [dim/2]
    pe = torch.zeros(length, dim)
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term)
    return pe

def create_2d_sin_cos_pos_emb(nh: int, nw: int, dim: int) -> torch.Tensor:
    assert dim % 2 == 0
    half = dim // 2
    pe_h = get_1d_sincos_pos_embed(nh, half)  # [nh, half]
    pe_w = get_1d_sincos_pos_embed(nw, half)  # [nw, half]
    # Combine by concatenation for each (h,w)
    pe_h_expand = pe_h.unsqueeze(1).expand(nh, nw, half)
    pe_w_expand = pe_w.unsqueeze(0).expand(nh, nw, half)
    pe_2d = torch.cat([pe_h_expand, pe_w_expand], dim=-1)  # [nh, nw, dim]
    return pe_2d.view(nh * nw, dim)

# Build tokens and pos-embeddings
img_tokens = patchify(img_tensor, patch_size)  # [T, D]
D = img_tokens.shape[1]
pos_emb = create_2d_sin_cos_pos_emb(nh, nw, D)  # [T, D]
img_tokens_pe = img_tokens + pos_emb

print({"tokens_shape": tuple(img_tokens.shape), "pos_emb_shape": tuple(pos_emb.shape)})


In [ ]:
# Compute attention with torch.nn.MultiheadAttention

# Tokens with positional embedding
tokens = img_tokens_pe  # [T, D]
T, D = tokens.shape

num_heads = 8  # 48 / 8 = 6 dims per head
assert D % num_heads == 0, "embed dim must be divisible by number of heads"

mha = nn.MultiheadAttention(embed_dim=D, num_heads=num_heads)

# MHA expects [T, B, D] by default
x = tokens.unsqueeze(1)  # [T, 1, D]
attn_out, attn_weights = mha(x, x, x, need_weights=True, average_attn_weights=False)

# attn_weights: [B, num_heads, T, T] (PyTorch >=2), or [num_heads, T, T] (older)
if attn_weights.dim() == 4:
    attn_weights = attn_weights[0]  # [num_heads, T, T]

# Select query index (default: center patch)
query_y = nh // 2
query_x = nw // 2
query_idx = int(query_y * nw + query_x)

attn_map = attn_weights[:, query_idx, :]  # [num_heads, T]
attn_map = attn_map.mean(0)  # [T]
attn_map_2d = attn_map.view(nh, nw)

# Normalize to [0,1]
attn_map_2d = attn_map_2d - attn_map_2d.min()
attn_map_2d = attn_map_2d / (attn_map_2d.max() + 1e-8)

print({"attn_shape": tuple(attn_weights.shape), "map_shape": tuple(attn_map_2d.shape), "query_idx": query_idx})


In [ ]:
# Visualize attention heatmap over the image

# Upsample attention map to image resolution
heat = attn_map_2d.unsqueeze(0).unsqueeze(0)  # [1,1,nh,nw]
heat_up = F.interpolate(heat, size=(H, W), mode="bilinear", align_corners=False)[0, 0]
heat_up_np = heat_up.numpy()

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(img_np)
ax.imshow(heat_up_np, cmap="jet", alpha=0.45, interpolation="bilinear")
ax.set_title(f"Attention heatmap (query=({query_y},{query_x}))")
ax.axis("off")
plt.show()
